# Import libraries: pandas (tables), NumPy (math), scikit-learn (ML)

In [188]:
import pandas as pd      # tables / dataframes
import numpy as np       # math, vectors, distances
from sklearn.neighbors import KNeighborsClassifier  # kNN model

# ✅ TASK 1 "Penguins kNN"

In [189]:
import pandas as pd

# Penguins kNN data (manually defined)
penguins = pd.DataFrame({
    "X1": [39.1, 18.7, 181, 3750],
    "X2": [50.2, 14.3, 218, 5700],
    "Q":  [39.5, 17.4, 186, 3800],
    "Min": [30, 10, 170, 3000],
    "Max": [60, 20, 230, 6000]
}, index=["Bill L", "Bill D", "Flip", "BM"])

penguins

,X1,X2,Q,Min,Max
Bill L,39.1,50.2,39.5,30,60
Bill D,18.7,14.3,17.4,10,20
Flip,181.0,218.0,186.0,170,230
BM,3750.0,5700.0,3800.0,3000,6000


Convert tabular data to numerical vectors

In [190]:
import numpy as np

# Convert table columns to numpy feature vectors
X1 = penguins["X1"].to_numpy()
X2 = penguins["X2"].to_numpy()
Q  = penguins["Q"].to_numpy()

X1, X2, Q

(array([  39.1,   18.7,  181. , 3750. ]),
 array([  50.2,   14.3,  218. , 5700. ]),
 array([  39.5,   17.4,  186. , 3800. ]))

In [191]:
X = np.vstack([X1, X2])        # training features
y = ["Adelie", "Gentoo"]       # labels
q = Q                           # query

a) Min–Max normalization to [0,1]

In [192]:
import numpy as np

# Original vectors
X1 = np.array([39.1, 18.7, 181, 3750])
X2 = np.array([50.2, 14.3, 218, 5700])
Q  = np.array([39.5, 17.4, 186, 3800])

# Min / Max values
min_vals = np.array([30, 10, 170, 3000])
max_vals = np.array([60, 20, 230, 6000])

# Min–Max normalization
X1_n = (X1 - min_vals) / (max_vals - min_vals)
X2_n = (X2 - min_vals) / (max_vals - min_vals)
Q_n  = (Q  - min_vals) / (max_vals - min_vals)

# Euclidean distance (explicit formula)
d_q_x1 = np.sqrt(np.sum((Q_n - X1_n)**2))
d_q_x2 = np.sqrt(np.sum((Q_n - X2_n)**2))

d_q_x1, d_q_x2

(np.float64(0.15588457268119896), np.float64(0.9533449882737448))

Construct normalized feature matrix

In [193]:
import pandas as pd

# Build normalized matrix as a table
norm_df = pd.DataFrame(
    [X1_n, X2_n, Q_n],
    index=["X1 (Adelie)", "X2 (Gentoo)", "Q (unknown)"],
    columns=["Bill L", "Bill D", "Flip", "BM"]
)

norm_df

,Bill L,Bill D,Flip,BM
X1 (Adelie),0.303333,0.87,0.183333,0.250000
X2 (Gentoo),0.673333,0.43,0.800000,0.900000
Q (unknown),0.316667,0.74,0.266667,0.266667


b) An appropriate global distance function is the Euclidean distance computed over all normalized numeric features, as it equally weights each feature and is suitable for continuous data.

In [194]:
import numpy as np
import pandas as pd

# --- Global distance function: Euclidean distance (on normalized features) ---
def euclidean_distance(a, b):
    """Compute Euclidean distance between two numeric vectors."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.sqrt(np.sum((a - b) ** 2))

# --- Distances from query Q to labelled examples X1 and X2 ---
d_q_x1 = euclidean_distance(Q_n, X1_n)
d_q_x2 = euclidean_distance(Q_n, X2_n)

print("d(Q, X1) =", d_q_x1)
print("d(Q, X2) =", d_q_x2)

# --- 1-NN decision: assign the label of the closest example ---
pred_label = "Adelie" if d_q_x1 < d_q_x2 else "Gentoo"
print("1-NN predicted label for Q:", pred_label)

d(Q, X1) = 0.15588457268119896
d(Q, X2) = 0.9533449882737448
1-NN predicted label for Q: Adelie


с) Based on Euclidean distance computed on normalized features, a 1-NN classifier assigns the query example the class label **Adelie**, as it is closer to X1 than to X2.

# ✅ TASK 2 "Pairwise Distances"

In [195]:
# Import libraries and prepare training data

import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Training data (from the table)
train = pd.DataFrame({
    "distance": [1.5, 2.8, 1.8, 2.9, 2.2, 3.0, 2.4, 3.2, 3.6],
    "class":    ["over","under","over","under","under","under","under","over","over"]
})

train

,distance,class
0,1.5,over
1,2.8,under
2,1.8,over
3,2.9,under
4,2.2,under
5,3.0,under
6,2.4,under
7,3.2,over
8,3.6,over


In [196]:
X = train[["distance"]].values   # feature matrix
y = train["class"].values        # labels

a) What class label would a 3-NN classifier assign to q? | **ANSWER - OVER**


In [197]:
knn_3 = KNeighborsClassifier(n_neighbors=3)
knn_3.fit(X, y)

# query q has distance = 0 to itself (we compare neighbours by distance feature)
q = np.array([[0]])

knn_3.predict(q)

array(['over'], dtype=object)

b) What class label would a 4-NN classifier assign to q? | **ANSWER - OVER**

In [198]:
knn_4 = KNeighborsClassifier(n_neighbors=4)
knn_4.fit(X, y)

knn_4.predict(q)

array(['over'], dtype=object)

c) What class label would a weighted 4-NN classifier assign to q? | **ANSWER - OVER**


In [199]:
knn_4_w = KNeighborsClassifier(n_neighbors=4, weights="distance")
knn_4_w.fit(X, y)

knn_4_w.predict(q)

array(['over'], dtype=object)

# ✅ TASK 3 "Car features"

In [200]:
import pandas as pd
import numpy as np

# Define the data for the two examples
data = {
    "Feature": ["Manufacturer", "Model", "Engine Size", "Fuel", "Mileage", "Condition"],
    "x1": ["Ford", "Fiesta", 1100, "Petrol", 65000, "Excellent"],
    "x2": ["Citroen", "C3", 1800, "Diesel", 37000, "Fair"]
}

df = pd.DataFrame(data)
df

,Feature,x1,x2
0,Manufacturer,Ford,Citroen
1,Model,Fiesta,C3
2,Engine Size,1100,1800
3,Fuel,Petrol,Diesel
4,Mileage,65000,37000
5,Condition,Excellent,Fair


a) The numeric features Engine Size and Mileage were normalized to the range [0,1] using min–max normalization based on the given feature ranges.

In [201]:
# Feature ranges
engine_min, engine_max = 1000, 3000
mileage_min, mileage_max = 1000, 100000

# Extract numeric values
engine_x1, engine_x2 = 1100, 1800
mileage_x1, mileage_x2 = 65000, 37000

# Min–Max normalization
engine_x1_n = (engine_x1 - engine_min) / (engine_max - engine_min)
engine_x2_n = (engine_x2 - engine_min) / (engine_max - engine_min)

mileage_x1_n = (mileage_x1 - mileage_min) / (mileage_max - mileage_min)
mileage_x2_n = (mileage_x2 - mileage_min) / (mileage_max - mileage_min)

engine_x1_n, engine_x2_n, mileage_x1_n, mileage_x2_n

(0.05, 0.4, 0.6464646464646465, 0.36363636363636365)

Encode and normalize ordinal feature (Condition)

In [202]:
# Ordinal encoding for Condition
condition_map = {
    "Poor": 0,
    "Fair": 1,
    "Good": 2,
    "Excellent": 3
}

# Encode
cond_x1 = condition_map["Excellent"]
cond_x2 = condition_map["Fair"]

# Normalize to [0,1]
cond_x1_n = cond_x1 / 3
cond_x2_n = cond_x2 / 3

cond_x1, cond_x2, cond_x1_n, cond_x2_n

(3, 1, 1.0, 0.3333333333333333)

Encode categorical features using binary mismatch

In [203]:
# Binary mismatch for categorical (nominal) features
def categorical_distance(a, b):
    return 0 if a == b else 1

# Manufacturer
d_manufacturer = categorical_distance("Ford", "Citroen")

# Model
d_model = categorical_distance("Fiesta", "C3")

# Fuel
d_fuel = categorical_distance("Petrol", "Diesel")

d_manufacturer, d_model, d_fuel

(1, 1, 1)

с) A suitable global distance function is the average of per-feature distances, using binary mismatch for categorical features, min–max–normalized absolute differences for numeric features, and normalized ordinal differences for the condition feature.

**ANSWER - The global distance between x1 and x2 is:** **0.716**

In [204]:
import numpy as np

# Individual distances (from previous calculations and kernel state):
# d_manufacturer = 1 (from yBS0BuyfsmT5)
# d_model = 1 (from yBS0BuyfsmT5)
# d_fuel = 1 (from yBS0BuyfsmT5)

# Calculate numeric and ordinal feature distances using normalized values
# engine_x1_n, engine_x2_n from Poe43VPBpmH4
d_engine = abs(engine_x1_n - engine_x2_n)

# mileage_x1_n, mileage_x2_n from Poe43VPBpmH4
d_mileage = abs(mileage_x1_n - mileage_x2_n)

# cond_x1_n, cond_x2_n from kZICVpk2rAJ0
d_condition = abs(cond_x1_n - cond_x2_n)

# global distance (average of all feature distances)
D_x1_x2 = np.mean([
    d_manufacturer,
    d_model,
    d_fuel,
    d_engine,
    d_mileage,
    d_condition
])

print(f"The global distance between x1 and x2 is: {D_x1_x2}")

The global distance between x1 and x2 is: 0.7165824915824915


# ✅ TASK 4 "Household"

*Please upload **"Household"** from Dataset_Tutorial_kNN.zip.*

In [205]:
import pandas as pd
from google.colab import files

# Upload CSV file
files.upload()   # choose Household.csv

# Read the table
house = pd.read_csv("Household.csv")
house

Saving Household.csv to Household (7).csv


,Household,Groceries,Education,Travel,Category
0,H1,2000,4000,500,C1
1,H2,3000,6000,1000,C1
2,H3,2000,2000,2000,C2
3,H4,3000,3000,3000,C2


In [206]:
# Labels (classes)
y = house["Category"].values

# Features (numeric only)
X = house[["Groceries", "Education", "Travel"]].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4, 3)
y shape: (4,)


In [207]:
q = np.array([[2500, 3500, 2000]])
print("q shape:", q.shape)

q shape: (1, 3)


The query example using 1-NN and Euclidean distance is **C2**

In [208]:
house_kNN = KNeighborsClassifier(n_neighbors=1)
house_kNN.fit(X, y)
print('Query is classified as', house_kNN.predict(q)[0])

Query is classified as C2


The query example using 1-NN and Euclidean distance is **C1**

In [209]:
house_kNN = KNeighborsClassifier(
    n_neighbors=1,
    metric="correlation",
    algorithm="brute"
)
house_kNN.fit(X, y)
print('Query is classified as', house_kNN.predict(q)[0])

Query is classified as C1


# ✅ TASK 5 "Athlete"

*Please upload **"AthleteSelection"** from Dataset_Tutorial_kNN.zip.*

In [210]:
import pandas as pd
import numpy as np
from google.colab import files

files.upload()

athlete = pd.read_csv("AthleteSelection.csv", index_col=0)
athlete.head()

Saving AthleteSelection.csv to AthleteSelection (2).csv


,Speed,Agility,Selected
Athlete,,,
x1,2.50,6.00,No
x2,3.75,8.00,No
x3,2.25,5.50,No
x4,3.25,8.25,No
x5,2.75,7.50,No


StandardScaler normalisation: **0., -0.169, 0.390**

In [211]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
q_scaled = scaler.transform(q)

In [212]:
X_scaled
q_scaled

array([[ 0.        , -0.16903085,  0.39056673]])

MinMaxScaler normalisation: **0.5, 0.375, 0.6**

In [213]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1)).fit(X)
X_scaled = scaler.transform(X)
q_scaled = scaler.transform(q)

In [214]:
X_scaled
q_scaled

array([[0.5  , 0.375, 0.6  ]])

# ✅ TASK 6 "Sepsis"

*Please upload **"s41598-020-73558-3_sepsis_survival_primary_cohort.csv"** from Dataset_Tutorial_kNN.zip.*



In [215]:
import pandas as pd
import numpy as np
from google.colab import files

files.upload()

Saving s41598-020-73558-3_sepsis_survival_primary_cohort.csv to s41598-020-73558-3_sepsis_survival_primary_cohort (4).csv


{'s41598-020-73558-3_sepsis_survival_primary_cohort (4).csv': b'age_years,sex_0male_1female,episode_number,hospital_outcome_1alive_0dead\n21,1,1,1\n20,1,1,1\n21,1,1,1\n77,0,1,1\n72,0,1,1\n83,0,1,1\n74,0,1,1\n74,1,1,1\n69,0,1,1\n53,1,1,1\n72,1,1,1\n82,0,1,1\n82,0,2,1\n83,0,3,1\n83,0,4,1\n75,1,1,1\n72,0,1,0\n45,0,1,1\n45,0,2,1\n56,1,1,1\n46,1,1,1\n48,1,1,1\n40,1,2,1\n39,0,1,1\n40,0,1,1\n56,0,1,1\n70,1,1,1\n47,1,1,1\n47,1,2,1\n46,0,1,1\n46,0,2,1\n27,1,1,1\n11,1,1,1\n91,0,1,1\n91,0,2,1\n91,0,3,1\n7,1,1,1\n79,0,1,1\n83,1,1,1\n84,1,2,1\n16,1,1,1\n73,0,1,1\n17,0,1,1\n17,0,2,1\n18,0,3,1\n63,0,1,0\n40,0,1,1\n73,0,1,1\n88,1,1,1\n89,1,2,0\n70,0,1,1\n84,0,1,1\n75,0,1,1\n75,0,2,1\n76,0,1,1\n76,0,2,1\n41,0,1,1\n70,1,1,1\n66,1,1,1\n79,0,1,1\n79,0,2,1\n80,0,3,0\n62,1,1,1\n62,1,2,1\n62,1,3,0\n20,0,1,1\n47,1,1,1\n59,1,1,1\n55,1,1,1\n55,1,2,1\n56,1,3,0\n68,0,1,1\n68,0,2,1\n79,0,1,1\n76,1,1,1\n76,1,2,1\n33,1,1,1\n48,0,1,1\n71,1,1,1\n48,0,1,1\n8,0,1,1\n47,0,1,1\n58,0,1,1\n53,1,1,1\n40,0,1,1\n41,0,2,1\n21,0

In [216]:
sepsis = pd.read_csv(
    "s41598-020-73558-3_sepsis_survival_primary_cohort.csv"
)

sepsis.head()

,age_years,sex_0male_1female,episode_number,hospital_outcome_1alive_0dead
0,21,1,1,1
1,20,1,1,1
2,21,1,1,1
3,77,0,1,1
4,72,0,1,1


In [217]:
# Target (label)
y = sepsis["hospital_outcome_1alive_0dead"].values

# Features
X_raw = sepsis.drop(columns=["hospital_outcome_1alive_0dead"])

X_raw.shape, y.shape

((110204, 3), (110204,))

In [218]:
# Split the dataset into training and test sets using a fixed random seed and scaled the features to ensure fair distance calculations for k-NN.

from sklearn.model_selection import train_test_split
from sklearn import preprocessing

# Split into train/test (same proportions as the notebook)
X_tr_raw, X_ts_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=1/3, random_state=42
)

# Scale using StandardScaler (fit on train, transform train+test)
scaler = preprocessing.StandardScaler().fit(X_tr_raw)
X_train = scaler.transform(X_tr_raw)
X_test  = scaler.transform(X_ts_raw)

X_train.shape, X_test.shape

((73469, 3), (36735, 3))

ball_tree **ANSWER: Time 3.47 | Accuracy 0.91**  

In [219]:
import time
from sklearn.neighbors import KNeighborsClassifier

Sep_kNN = KNeighborsClassifier(n_neighbors=3, algorithm="ball_tree")
Sep_kNN.fit(X_train, y_train)

t_start = time.time()
acc = Sep_kNN.score(X_test, y_test)
t = time.time() - t_start

print(f"ball_tree | Time: {t:.4f}s | Accuracy: {acc:.4f}")

ball_tree | Time: 1.0072s | Accuracy: 0.9071


kd_tree **ANSWER: Time 2.21 | Accuracy 0.91**

In [220]:
import time
from sklearn.neighbors import KNeighborsClassifier

Sep_kNN = KNeighborsClassifier(n_neighbors=3, algorithm="kd_tree")
Sep_kNN.fit(X_train, y_train)

t_start = time.time()
acc = Sep_kNN.score(X_test, y_test)
t = time.time() - t_start

print(f"kd_tree  | Time: {t:.4f}s | Accuracy: {acc:.4f}")

kd_tree  | Time: 0.8100s | Accuracy: 0.9071


brute **ANSWER: Time 26.34 | Accuracy 0.92**

In [221]:
import time
from sklearn.neighbors import KNeighborsClassifier

Sep_kNN = KNeighborsClassifier(n_neighbors=3, algorithm="brute")
Sep_kNN.fit(X_train, y_train)

t_start = time.time()
acc = Sep_kNN.score(X_test, y_test)
t = time.time() - t_start

print(f"brute     | Time: {t:.4f}s | Accuracy: {acc:.4f}")

brute     | Time: 15.4968s | Accuracy: 0.9165
